Internal benchmark: source-associated difficulty...


In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels

def locate_repository():
    import os
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        if (parent / 'data/limfmed_dataset_gold.csv').is_file() and (parent / 'experiments').is_dir():
            if os.name == 'nt' and (not str(parent).startswith(chr(92) * 2 + '?' + chr(92))):
                return Path(chr(92) * 2 + '?' + chr(92) + str(parent))
            return parent
    raise FileNotFoundError('Run the notebook from inside the repository.')
ROOT = locate_repository()
OUTPUT_PATH = ROOT / 'experiments/external_benchmarks/medmcqa/internal_source_difficulty_analysis.json'
SOURCE_LABELS = {'AI': 'LLM-generated', 'checker': 'Symptom-checker audit', 'msd': 'MSD Manual'}
ARMS = {'gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/baseline_noroles_gpt_judge.csv', 'qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/baseline_noroles_qwen_judge.csv', 'deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/baseline_noroles_deepseekv4_judge.csv', 'deepseekv4pro': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4pro/llm_as_judge/baseline_noroles_deepseekv4pro_judge.csv', 'gemini': 'experiments/clinical_protocols/single_model_baselines/baseline_gemini/llm_as_judge/baseline_noroles_gemini_judge.csv', 'glm': 'experiments/clinical_protocols/single_model_baselines/baseline_glm/llm_as_judge/baseline_noroles_glm_judge.csv', 'kimi': 'experiments/clinical_protocols/single_model_baselines/baseline_kimi/llm_as_judge/baseline_noroles_kimi_judge.csv', 'P-base': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'P-base/org-ds': 'experiments/clinical_protocols/delphi_panels/heterogeneous_deepseek_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'P-base/org-qwen': 'experiments/clinical_protocols/delphi_panels/heterogeneous_qwen_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'P-base/org-gemini': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gemini_external_organizer/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/delphi_pweak_noroles_judge.csv', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/delphi_pmixed_noroles_judge.csv', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/delphi_pstrong_noroles_judge.csv', 'homog gpt': 'experiments/homogeneous_panels/three_gpt_members/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog qwen': 'experiments/homogeneous_panels/three_qwen_members/llm_as_judge/delphi_homog_qwen_noroles_judge.csv', 'homog deepseekv4': 'experiments/homogeneous_panels/three_deepseekv4_members_complete/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv', 'homog gemini': 'experiments/homogeneous_panels/three_gemini_members/llm_as_judge/delphi_homog_gemini_noroles_judge.csv'}
print(f'Read-only source root: {ROOT}')
print(f'Primary systems: {len(ARMS)}')


Read-only source root: .
Primary systems: 18


In [2]:
frames = []
provenance = []
for system, relative_path in ARMS.items():
    path = ROOT / relative_path
    data = pd.read_csv(path, sep=';', encoding='utf-8-sig', low_memory=False)
    data.columns = [str(c).lstrip('\ufeff') for c in data.columns]
    if 'table_source' in data.columns:
        data = data[data['table_source'].fillna('result_df') == 'result_df']
    data = data.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
    data['Judge_score'] = pd.to_numeric(data['Judge_score'], errors='coerce')
    data = data.dropna(subset=['Judge_score'])
    frame = data[['Vignette', 'Genealogy', 'Judge_score']].rename(columns={'Vignette': 'case', 'Genealogy': 'source', 'Judge_score': 'score'})
    frame['system'] = system
    if not set(frame['source']).issubset(SOURCE_LABELS):
        raise ValueError(f'Unexpected source label in {system}')
    frames.append(frame)
    provenance.append({'system': system, 'path': relative_path, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'scored_protocols': len(frame)})
long = pd.concat(frames, ignore_index=True)
print(f"Scored protocols: {len(long)}; unique cases: {long['case'].nunique()}")
print(long.groupby('source')['score'].agg(['size', 'mean', 'std']).to_string())


Scored protocols: 4113; unique cases: 229
         size      mean       std
source                           
AI       2717  0.702056  0.182548
checker   774  0.637267  0.209268
msd       622  0.714531  0.192151


In [3]:
pooled = []
for source, part in long.groupby('source', sort=True):
    pooled.append({'source': source, 'label': SOURCE_LABELS[source], 'cases': int(part['case'].nunique()), 'scored_protocols': len(part), 'mean_score': float(part['score'].mean()), 'sd_score': float(part['score'].std(ddof=1))})
per_system = []
for system, part in long.groupby('system', sort=False):
    groups = [block['score'].to_numpy() for _, block in part.groupby('source', sort=True)]
    test = stats.kruskal(*groups)
    source_results = []
    for source, block in part.groupby('source', sort=True):
        source_results.append({'source': source, 'label': SOURCE_LABELS[source], 'cases': len(block), 'mean_score': float(block['score'].mean()), 'sd_score': float(block['score'].std(ddof=1))})
    per_system.append({'system': system, 'scored_protocols': len(part), 'sources': source_results, 'kruskal_wallis_H': float(test.statistic), 'kruskal_wallis_p_uncorrected': float(test.pvalue)})
variance_models = {}
for predictor in ['source', 'case', 'system']:
    fit = smf.ols(f'score ~ C({predictor})', long).fit()
    variance_models[predictor] = {'n': int(fit.nobs), 'levels': int(long[predictor].nunique()), 'r_squared': float(fit.rsquared), 'adjusted_r_squared': float(fit.rsquared_adj)}
print(json.dumps(variance_models, indent=2))


{
  "source": {
    "n": 4113,
    "levels": 3,
    "r_squared": 0.019367615905802382,
    "adjusted_r_squared": 0.01889042253154727
  },
  "case": {
    "n": 4113,
    "levels": 229,
    "r_squared": 0.3461474418501811,
    "adjusted_r_squared": 0.3077647479114173
  },
  "system": {
    "n": 4113,
    "levels": 18,
    "r_squared": 0.32085213022976056,
    "adjusted_r_squared": 0.3180327129437791
  }
}


In [4]:
model = smf.ols("score ~ C(system) + C(source, Treatment(reference='AI'))", long).fit(cov_type='cluster', cov_kwds={'groups': long['case']}, use_t=False)
intervals = model.conf_int(alpha=0.05)
contrasts = []
source_terms = [term for term in model.params.index if 'C(source,' in term]
for term in source_terms:
    source = term.rsplit('[T.', 1)[1].rstrip(']')
    contrasts.append({'source': source, 'reference': 'AI', 'coefficient': float(model.params[term]), 'ci_lower': float(intervals.loc[term, 0]), 'ci_upper': float(intervals.loc[term, 1]), 'p_value': float(model.pvalues[term]), 'standard_error': float(model.bse[term])})
restriction = np.zeros((len(source_terms), len(model.params)))
for row, term in enumerate(source_terms):
    restriction[row, model.params.index.get_loc(term)] = 1.0
omnibus = model.f_test(restriction)
adjusted = {'formula': 'score ~ C(system) + C(source, reference=AI)', 'covariance': 'vignette-clustered; default small-sample covariance correction', 'interval_method': 'two-sided normal-based Wald 95% interval', 'n_protocols': int(model.nobs), 'n_clusters': int(long['case'].nunique()), 'contrasts': contrasts, 'omnibus': {'F': float(np.asarray(omnibus.fvalue).item()), 'df_num': float(omnibus.df_num), 'df_denom': float(omnibus.df_denom), 'p_value': float(np.asarray(omnibus.pvalue).item())}}
print(json.dumps(adjusted, indent=2))


{
  "formula": "score ~ C(system) + C(source, reference=AI)",
  "covariance": "vignette-clustered; default small-sample covariance correction",
  "interval_method": "two-sided normal-based Wald 95% interval",
  "n_protocols": 4113,
  "n_clusters": 229,
  "contrasts": [
    {
      "source": "checker",
      "reference": "AI",
      "coefficient": -0.06475413685269354,
      "ci_lower": -0.10244891782520901,
      "ci_upper": -0.02705935588017807,
      "p_value": 0.0007600934205531983,
      "standard_error": 0.01923238450800479
    },
    {
      "source": "msd",
      "reference": "AI",
      "coefficient": 0.011353981526208656,
      "ci_lower": -0.03306743063202393,
      "ci_upper": 0.055775393684441246,
      "p_value": 0.616398573452304,
      "standard_error": 0.0226644022587267
    }
  ],
  "omnibus": {
    "F": 6.254790737813845,
    "df_num": 2.0,
    "df_denom": 228.0,
    "p_value": 0.0022671535979480275
  }
}


In [5]:
reference_systems = ['P-base', 'gpt', 'qwen', 'deepseekv4']
wide = long[long['system'].isin(reference_systems)].pivot(index='case', columns='system', values='score')
wide = wide.dropna(subset=reference_systems)
case_sources = long.drop_duplicates('case').set_index('case')['source']
wide['source'] = case_sources.reindex(wide.index)
wide['panel_minus_best_member'] = wide['P-base'] - wide[['gpt', 'qwen', 'deepseekv4']].max(axis=1)
advantage_groups = [part['panel_minus_best_member'].to_numpy() for _, part in wide.groupby('source', sort=True)]
advantage_test = stats.kruskal(*advantage_groups)
advantage = {'outcome': 'P-base minus highest-scoring independent member for each case', 'common_cases': len(wide), 'sources': [{'source': source, 'cases': len(part), 'mean_difference': float(part['panel_minus_best_member'].mean())} for source, part in wide.groupby('source', sort=True)], 'kruskal_wallis_H': float(advantage_test.statistic), 'kruskal_wallis_p': float(advantage_test.pvalue)}
print(json.dumps(advantage, indent=2))


{
  "outcome": "P-base minus highest-scoring independent member for each case",
  "common_cases": 228,
  "sources": [
    {
      "source": "AI",
      "cases": 151,
      "mean_difference": 0.0874675496688742
    },
    {
      "source": "checker",
      "cases": 43,
      "mean_difference": 0.10345116279069769
    },
    {
      "source": "msd",
      "cases": 34,
      "mean_difference": 0.07277058823529411
    }
  ],
  "kruskal_wallis_H": 1.1783404857480122,
  "kruskal_wallis_p": 0.5547874326282445
}


In [6]:
report = {'analysis': 'Internal benchmark source-associated difficulty; completed primary no-role clinical runs', 'created_date': '2026-10-11', 'timezone': 'Europe/Moscow', 'n_systems': len(ARMS), 'n_single_model_baselines': 7, 'n_panel_configurations': 11, 'n_scored_protocols': len(long), 'n_unique_cases': int(long['case'].nunique()), 'input_columns': ['Vignette', 'Genealogy', 'Judge_score', 'table_source (if present)'], 'source_codes': SOURCE_LABELS, 'interpretation': ['Difficulty is operationalized as rubric performance; it is not a direct clinical-complexity rating.', 'Source explains only a small share of total pooled score variance, but this does not establish equivalent source difficulty.', 'Audit cases receive lower mean scores in all 18 primary systems.', 'A nonsignificant test of panel-advantage heterogeneity is distinct from a test of source-associated difficulty.', 'Per-system Kruskal-Wallis p values are descriptive and not multiplicity-adjusted.', 'Pooled source means weight each available scored protocol equally; source groups differ in size.'], 'pooled_source_results': pooled, 'per_system_source_results': per_system, 'separate_categorical_variance_models': variance_models, 'system_adjusted_source_model': adjusted, 'reference_panel_advantage_by_source': advantage, 'provenance': provenance, 'software': {'python': __import__('sys').version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__, 'statsmodels': statsmodels.__version__}}
OUTPUT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
print(f'Saved analysis JSON: {OUTPUT_PATH}')


Saved analysis JSON: .\experiments\external_benchmarks\medmcqa\internal_source_difficulty_analysis.json
